In [ ]:
import pandas as pd
import os
import numpy as np
import re
from IPython.display import display # Needed for the .display() calls

import pickle
from pathlib import Path


In [ ]:
data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-12-09_preparing_cohort_phenotype_files_for_regenie_input"


results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-09_preparing_cohort_phenotype_files_for_regenie_input"
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-03_using_matchit_to_match_case_and_control_cohorts"

scratch = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-09_preparing_cohort_phenotype_files_for_regenie_input"

!mkdir {scratch}

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
def flatten_ancestry_pcs_df(pc_df): 
    
    
    pc_df["pca_features"] = pc_df["pca_features"].str[1:-1]
    
    PCs = pc_df["pca_features"].str.split(",", n = 16, expand = True)
    PCs = PCs.astype(float)
    
    pid = pc_df[["research_id"]]
    
    columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
    PCs.columns = columns
    
    PCs_final = pd.concat([pid, PCs], axis = 1)
    
    PCs_final.to_csv('wrangled_ancestry_pcs.csv')
    
    return PCs_final
    

In [ ]:
# ------------------------------------------------------------------
#  PC normalisation helper (as before)
# ------------------------------------------------------------------
def normalize_pc_columns(pcs_df, pc_id_col, prefix="PC"):
    """Rename variants of PC columns to 'PC<k>' (pc1, PC_01, 'PC 2' -> PC1/PC2), keep only ID + PCs."""
    df = pcs_df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        if c == pc_id_col:
            continue
        s = str(c).strip()
        # match pc, PC, PC_, PC<space> and zero-padded numbers
        m = re.match(r'(?i)^pc[\s_]*0*([0-9]+)$', s)
        if not m:
            # also handle 'principal component 1' / 'pcscore1' styles if present
            m = re.match(r'(?i)^(principal[\s_]*component|pcscore)[\s_]*0*([0-9]+)$', s)
            if m:
                num = m.group(2)
                rename[c] = f'PC{int(num)}'
                continue
        if m:
            num = m.group(1)
            rename[c] = f'PC{int(num)}'
    df = df.rename(columns=rename)

    # keep ID + PC* columns only
    pc_cols = [c for c in df.columns if c != pc_id_col and str(c).upper().startswith(prefix.upper())]
    # ensure numeric PCs
    for c in pc_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # sort PC columns in numeric order (PC1, PC2, ..., PC10)
    def pc_num(name):
        m = re.search(r'(\d+)$', name)
        return int(m.group(1)) if m else 10**9

    pc_cols = sorted(pc_cols, key=pc_num)
    return df[[pc_id_col] + pc_cols], pc_cols


# ------------------------------------------------------------------
#  Main function: single df + optional external PC file
# ------------------------------------------------------------------
def create_analysis_files_single(
    df,
    pcs_df=None,                # separate PC table (optional)
    id_col="person_id",
    sex_col="sex_at_birth",
    age_col="age",
    phenotype_col="case",       # 0/1 column in df
    pc_id_col="research_id",    # ID column *in pcs_df*
    pc_prefix="PC",

    phenotype_name="phenotype",
    pheno_outfile="pheno.tsv",
    covar_outfile="covar.tsv",
    merged_outfile="merged.tsv",
    variants_outfile="variants.txt",
    variant_list=None,

    index_date=None,
    write_files=True,
    show_previews=True
):
    if variant_list is None:
        variant_list = []

    # ---------- 1) Base IDs ----------
    data = df.copy()
    data[id_col] = data[id_col].astype("string")

    ids = pd.Series(data[id_col].dropna().unique(), dtype="string", name="IID")
    base = pd.DataFrame({"IID": ids})
    base["FID"] = base["IID"]
    base = base.astype({"FID": "string", "IID": "string"})[["FID", "IID"]]

    # ---------- 2) Phenotype ----------
    pheno = base.copy()

    ph = pd.to_numeric(data.set_index(id_col)[phenotype_col], errors="coerce")
    ph = ph.reindex(pheno["IID"])
    ph_clean = np.where(ph == 1, 1, np.where(ph == 0, 0, np.nan))
    pheno[phenotype_name] = ph_clean

    if write_files:
        pheno.to_csv(pheno_outfile, sep="\t", index=False)

    # ---------- 3) Covariates: sex / age ----------
    dem = (data[[id_col, sex_col, age_col]]
           .rename(columns={id_col: "IID", sex_col: "sex", age_col: "age"})
           .drop_duplicates(subset=["IID"]))

    dem["IID"] = dem["IID"].astype("string")
    dem["FID"] = dem["IID"]

    dem["sex"] = (dem["sex"].astype(str).str.strip().str.upper()
                    .replace({"MALE": "M",
                              "FEMALE": "F",
                              "1": "M",
                              "2": "F",
                              "0": np.nan,
                              "UNKNOWN": np.nan}))

    dem["age"] = pd.to_numeric(dem["age"], errors="coerce")

    covar = dem.copy()
    pc_cols_found = []

    # ---------- 4) PCs: from separate pcs_df if provided, otherwise from df ----------
    if pcs_df is not None:
        pcs_df = pcs_df.copy()
        pcs_df[pc_id_col] = pcs_df[pc_id_col].astype("string")

        pcs_norm, pc_cols_found = normalize_pc_columns(
            pcs_df, pc_id_col=pc_id_col, prefix=pc_prefix
        )
        pcs_norm = pcs_norm.rename(columns={pc_id_col: "IID"})
        pcs_norm["IID"] = pcs_norm["IID"].astype("string")

        covar = covar.merge(pcs_norm, on="IID", how="left")

    else:
        # Look for PC columns already in df
        pc_cols_in_df = [c for c in data.columns
                         if c != id_col and str(c).upper().startswith(pc_prefix.upper())]
        if pc_cols_in_df:
            pcs_input = data[[id_col] + pc_cols_in_df].drop_duplicates(subset=[id_col])
            pcs_norm, pc_cols_found = normalize_pc_columns(
                pcs_input, pc_id_col=id_col, prefix=pc_prefix
            )
            pcs_norm = pcs_norm.rename(columns={id_col: "IID"})
            pcs_norm["IID"] = pcs_norm["IID"].astype("string")
            covar = covar.merge(pcs_norm, on="IID", how="left")

    # ---------- 5) Order columns ----------
    def pc_key(name):
        m = re.search(r'(\d+)$', name)
        return (int(m.group(1)) if m else 10**9, name)

    pc_order = sorted(
        [c for c in covar.columns if c.upper().startswith(pc_prefix.upper())],
        key=pc_key
    )

    wanted_cols = ["FID", "IID", "sex", "age"] + pc_order
    covar = base.merge(
        covar[[c for c in wanted_cols if c in covar.columns]],
        on=["FID", "IID"],
        how="left"
    )

    if write_files:
        covar.to_csv(covar_outfile, sep="\t", index=False)

    # ---------- 6) One-file version ----------
    merged = covar.merge(
        pheno[["FID", "IID", phenotype_name]],
        on=["FID", "IID"],
        how="left"
    )

    if write_files:
        merged.to_csv(merged_outfile, sep="\t", index=False)

    # ---------- 7) Variant list ----------
    if write_files and variant_list:
        with open(variants_outfile, "w") as f:
            for v in variant_list:
                f.write(str(v).strip() + "\n")

    # ---------- 8) Diagnostics ----------
    if show_previews:
        if index_date is not None:
            try:
                print("Index date used:", index_date.date())
            except Exception:
                print(f"Index date provided: {index_date}")

        print("PC columns detected & merged:", pc_cols_found if pc_cols_found else "None found")

        base_covar_n = ((covar["sex"].notna()) | (covar["age"].notna())).sum()
        if pc_order:
            pc_overlap_n = covar[pc_order].notna().any(axis=1).sum()
        else:
            pc_overlap_n = 0

        print(f"Samples in base: {len(base)}"
              f"  | with any demo data (sex/age): {base_covar_n}"
              f"  | with PCs merged: {pc_overlap_n}")

        n_cases = int((pheno[phenotype_name] == 1).sum())
        n_ctrls = int((pheno[phenotype_name] == 0).sum())
        n_missing = int(pheno[phenotype_name].isna().sum())

        print("N total:", len(base),
              "| cases:", n_cases,
              "| controls:", n_ctrls,
              "| missing pheno:", n_missing)

    return pheno, covar, merged


In [ ]:
def batch_process(
    folders,
    pc_file_path,
    file_suffix=".csv",          # pattern for your phenotype files
    id_col="person_id",
    phenotype_col="case",        # 0/1 col in each CSV
    sex_col="sex_at_birth",
    age_col="age",
    pc_id_col="research_id",     # ID col in the PC file
    pc_prefix="PC",
    variant_list=None
):
    """
    Loop over multiple folders; in each folder, process all CSV files
    that match *file_suffix using create_analysis_files_single().
    """
    if variant_list is None:
        variant_list = []

    pc_file_path = Path(pc_file_path)
    print(f"Loading shared PC file from: {pc_file_path}")
    pcs_df_main = pd.read_csv(pc_file_path)

    for folder in folders:
        folder = Path(folder)

        if not folder.exists():
            print(f"[!] Folder does not exist, skipping: {folder}")
            continue

        files = list(folder.glob(f"*{file_suffix}"))
        if not files:
            print(f"No files in {folder} matching *{file_suffix}")
            continue

        print(f"\nFound {len(files)} files in {folder}.")

        for file_path in files:
            # phenotype name from filename (without suffix)
            pheno_name = file_path.name.replace(file_suffix, "")

            print("\n" + "="*50)
            print(f"Processing phenotype: {pheno_name}")
            print(f"Loading file: {file_path}")

            try:
                df = pd.read_csv(file_path)

                # unique outputs for this phenotype,
                # written back into the same folder
                pheno_out    =  f"{scratch}/{pheno_name}_pheno.tsv"
                covar_out    =  f"{scratch}/{pheno_name}_covar.tsv"
                merged_out   =  f"{scratch}/{pheno_name}_merged.tsv"
                variants_out = f"{scratch}/{pheno_name}_variants.txt"

                print(f"  Running analysis for {pheno_name}...")

                pheno_data, covar_data, merged_data = create_analysis_files_single(
                    df=df,
                    pcs_df=pcs_df_main,      # shared PC file
                    id_col=id_col,
                    sex_col=sex_col,
                    age_col=age_col,
                    phenotype_col=phenotype_col,
                    pc_id_col=pc_id_col,
                    pc_prefix=pc_prefix,
                    phenotype_name=pheno_name,
                    pheno_outfile=str(pheno_out),
                    covar_outfile=str(covar_out),
                    merged_outfile=str(merged_out),
                    variants_outfile=str(variants_out),
                    variant_list=variant_list,
                    write_files=True,
                    show_previews=False      # set True if you want spammy previews
                )

                print(f"  [+] Success: Finished processing for {pheno_name}.")

            except Exception as e:
                print(f"  [!] ERROR processing {pheno_name}: {e}")

    print("\n" + "="*50)
    print("All phenotype processing complete.")


In [ ]:

def concatenate_tsvs(input_dir):
    input_dir = Path(input_dir)
    dfs = []
    for f in input_dir.glob("*_merged.tsv"):
        dfs.append(pd.read_csv(f, sep="\t"))
    result = pd.concat(dfs,  axis = 0,  ignore_index=True)
#    result.to_csv(output_path, sep="\t", index=False)
    return result




In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
##function calls

'''
raw_ancestry_pcs = get_data_pkl(data, 'raw_ancestry_pcs.pkl')
    
wrangled_pcs = flatten_ancestry_pcs_df(raw_ancestry_pcs)


folders_to_process = [
    Path(f"{results1}/b1_matched"),
    Path(f"{results1}/c1_matched"),
    Path(f"{results1}/covid_matched"),
    Path(f"{results1}/d1_matched"),
    Path(f"{results1}/flu_matched"),
    Path(f"{results1}/ns_matched"),
    Path(f"{results1}/ns_raw_vax_matched"),
    Path(f"{results1}/ns_vax_matched"),
]


batch_process(
    folders=folders_to_process,
    pc_file_path=Path(f"{results}/wrangled_ancestry_pcs.csv"),
    file_suffix=".csv",          # or "_cohort.csv", etc.
    id_col="person_id",
    phenotype_col="case",        # your 0/1 column
    sex_col="sex",
    age_col="age",
    pc_id_col="research_id",
    pc_prefix="PC",
#    variant_list=["rs17569141"]
)

'''



#concat_pheno_file = concatenate_tsvs(Path(f"{scratch}"))
                                     

#concat_pheno_file = concatenate_tsvs(Path(f"{scratch}"), Path(f"{results}/all_matched_concatenated.tsv"))

create_df_pkl(concat_pheno_file, f"{results}/all_matched_concatenated_tsv.pkl")





In [ ]:
##visulaize

#wrangled_pcs


df = concat_pheno_file[concat_pheno_file["matched_human_immunodeficiency_virus_infection_matchit_cohort"] == 1]


In [ ]:
df